# 3D polymer + loop extrusion

Set up and run a polychrom/OpenMM simulation of a polymer with cohesin loop extrusion
and CTCF boundaries. Work top to bottom: each cell defines one group of settings, the
last few assemble them, run, and look at the output.

The code lives in three modules next to this notebook:

| file | what it does |
| --- | --- |
| `extrusion.py` | CTCF stall arrays and the 1D LEF translocator (no OpenMM needed) |
| `sim3D.py` | `SimParams` + the polymer/force setup + the run loop |
| `smcBondUpdater.py` | pushes LEF positions into OpenMM harmonic bonds each step |

Compartments are **not** modelled. If you want type-specific interactions you give
`monomer_types` and `interaction_matrix` directly (step 3) and they go straight to
polychrom's `heteropolymer_SSW`; leave them as `None` for a plain homopolymer.

In [1]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt

from polysim import extrusion, OUTPUTS
from polysim.sim3d import SimParams, run

# the first import compiles LEF_Dynamics.pyx via pyximport -- a few seconds, once
print("output goes to", OUTPUTS)


output goes to /mnt/md1/jjusuf/polysim/outputs


In [ ]:
npoly = 70000

params = SimParams(
    npoly=npoly,  # total number of monomers; ignored if chr_sizes is given
    density=0.3,  # number of monomers per unit volume

    # --- CTCF sites ---
    # a periodic array of CTCF sites
    # IMPORTANT: LEFT means stalls the left side of a LEF ("right-pointing")
    ctcf_left  = extrusion.tile_sites([200, 330, 724, 1425, 1433, 1604],
                                  period=2000, length=npoly),
    ctcf_right = extrusion.tile_sites([574, 694, 866, 1241, 1390, 1580, 1752, 1800], 
                                  period=2000, length=npoly),
                                  
    stall=0.8,
    stallall=False,  # True = stall everywhere; ignores the lists below
                     # IMPROVE THIS DESCRIPTION
    
    # --- loop extrusion ---
    life=75000,            # LEF lifetime, in LEF timesteps
    sep=240,               # monomers per LEF -> n_lefs = npoly // sep
    vlef=0.0025,           # p(step per leg per timestep)
    lifebooststalled=4,    # lifetime multiplier while stalled at a CTCF

    monomer_types=None,
    interaction_matrix=None,

    # --- integration ---
    platform="CUDA", gpu="0",
    integrator="langevin",
    dt=40,
    colrate=0.01,
    colrate0=0.01,
    poly_steps_per_lef_timestep=50,  # polymer timesteps per LEF timestep
                                     # from calibration run with default parameters, this causes LEF timestep to be ~20 ms

    # --- schedule ---
    numsave=36000,         # total number of save-blocks (100 hours)
    saveevery=500,         # number of blocks between saves (10 seconds); must divide blocks_per_updater, whose default value is 1000
    initskip=3000,         # blocks of equilibration for 3D polymer (1 minute)
    initsteps=2000000,     # LEF-only steps before the polymer starts moving (~11 hours)

    # --- output ---
    outpath=OUTPUTS,     # polysim.OUTPUTS, outside the repository
    flag="",             # label appended to the auto-generated folder name
)

print(params.summary())

polymer      70000 monomers in 1 chain(s) [70000]
confinement  sphere r=61.6 at density 0.3
interactions homopolymer (soft repulsion, 3.0 kT)
LEFs         291 (1 per 240 monomers), lifetime 75000, v=0.0025/step
CTCF         490 stall site(s), p=0.8 per encounter, lifetime x4 while stalled
schedule     39000 blocks written (3000 of them equilibration, drop those), 500 LEF steps each, 50 polymer steps per LEF step


## 5. Run


In [7]:
folder = run(params)

polymer      10000 monomers in 1 chain(s) [10000]
confinement  sphere r=36.8 at density 0.2
interactions homopolymer (soft repulsion, 3.0 kT)
LEFs         20 (1 per 480 monomers), lifetime 3000, v=0.05/step
CTCF         20 stall site(s), p=0.8 per encounter, lifetime x4 while stalled
schedule     10080 blocks written (80 of them equilibration, drop those), 100 LEF steps each, 450 polymer steps per LEF step

output -> outputs/trajectory0002_npoly10000_nchr1_dens0.2_life3000_sep480_vlef0.05_dt40_stallsites0.8_lifeboost4
equilibrating LEF dynamics for 1000000 steps...
  done in 0.4 s
updater init 0 / 1008  (equilibration)


INFO:root:Performing local energy minimization
INFO:root:adding force spherical_confinement 0
INFO:root:adding force harmonic_bonds 1
INFO:root:adding force angle 2
INFO:root:adding force polynomial_repulsive 3


Exclude neighbouring chain particles from polynomial_repulsive
Number of exceptions: 9999


INFO:root:Particles loaded. Potential energy is 16.158338
INFO:root:before minimization eK=1.5094171537274208, eP=16.15833820045713, time=0.0 ps
INFO:root:Particles loaded. Potential energy is 0.151255
INFO:root:after minimization eK=1.5706186306286698, eP=0.11686214986950959, time=0.0 ps
INFO:root:block    0 pos[1]=[-7.3 -5.0 -2.3] dr=8.77 t=1800.0ps kin=1.54 pot=1.36 Rg=16.971 SPS=20677 
INFO:root:block    1 pos[1]=[-15.7 -2.2 -7.0] dr=8.35 t=3600.0ps kin=1.52 pot=1.34 Rg=16.901 SPS=20431 
INFO:root:block    2 pos[1]=[-5.3 -9.6 3.0] dr=8.12 t=5400.0ps kin=1.53 pot=1.33 Rg=17.046 SPS=20418 
INFO:root:block    3 pos[1]=[-8.4 -4.3 6.1] dr=8.36 t=7200.0ps kin=1.54 pot=1.31 Rg=17.000 SPS=20659 
INFO:root:block    4 pos[1]=[-16.6 -16.5 0.2] dr=8.28 t=9000.0ps kin=1.53 pot=1.33 Rg=16.959 SPS=20728 
INFO:root:block    5 pos[1]=[-12.5 -5.8 -4.2] dr=8.59 t=10800.0ps kin=1.52 pot=1.30 Rg=16.927 SPS=20750 
INFO:root:block    6 pos[1]=[-14.0 -6.0 -0.9] dr=8.09 t=12600.0ps kin=1.50 pot=1.33 Rg=16.

updater init 1 / 1008  (equilibration)


...

updater init 2 / 1008  (equilibration)


INFO:root:adding force spherical_confinement 0
INFO:root:adding force harmonic_bonds 1
INFO:root:adding force angle 2
INFO:root:adding force polynomial_repulsive 3


Exclude neighbouring chain particles from polynomial_repulsive
Number of exceptions: 9999


INFO:root:Particles loaded. Potential energy is 1.346614
INFO:root:block    0 pos[1]=[1.6 14.5 9.3] dr=8.32 t=1800.0ps kin=1.53 pot=1.34 Rg=16.921 SPS=20873 
INFO:root:block    1 pos[1]=[-5.9 15.7 8.8] dr=8.34 t=3600.0ps kin=1.54 pot=1.33 Rg=16.988 SPS=20450 
/mnt/md0/jjusuf/miniconda3/envs/polysim3/lib/python3.12/site-packages/openmm/openmm.py:4700: SyntaxWarning: invalid escape sequence '\S'
  match = re.search("<([^?]\S*)", inputString)


KeyboardInterrupt: 

## 6. Look at the output

The folder holds:

| file | contents |
| --- | --- |
| `blocks_*.h5` | the trajectory, read with `polychrom.hdf5_format`; each block also carries an `SMCs` `(n_lefs, 2)` array of LEF leg positions, if `save_smc_bonds` |
| `paramsDict.pkl` | the `SimParams` as a dict |
| `sites.npz` | the per-monomer LEF arrays actually used (birth/death/stall/pause) |
| `bondsAdded.txt` | LEF steps taken / new bonds added over time, if `save_smc_bonds` |

Remember the first `n_equil_blocks` blocks are equilibration — drop them before analysis.

In [ ]:
from polychrom.hdf5_format import list_URIs, load_URI
import polychrom.polymer_analyses as polymer_analyses

uris = list_URIs(folder)
n_equil = params.schedule()["n_equil_blocks"]
print("{0} blocks written, dropping the first {1} (equilibration)".format(len(uris), n_equil))

production = uris[n_equil:]
last = load_URI(production[-1])["pos"]
print("last conformation:", last.shape)

In [ ]:
# radius of gyration over the production blocks, and the final conformation
rg = [np.sqrt(polymer_analyses.Rg2(load_URI(u)["pos"])) for u in production]

fig = plt.figure(figsize=(11, 4.5))

ax = fig.add_subplot(1, 2, 1)
ax.plot(rg, lw=1)
ax.set_xlabel("production block")
ax.set_ylabel("radius of gyration")
ax.set_title("equilibration check")

ax = fig.add_subplot(1, 2, 2, projection="3d")
for start, end, _ in params.chains:
    ax.plot(*last[start:end].T, lw=0.4)
ax.set_title("final conformation")
ax.set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
# the LEF arrays that were actually used -- a good sanity check on the CTCF setup
sites = np.load(os.path.join(folder, "sites.npz"))

fig, ax = plt.subplots(2, 1, figsize=(11, 4), sharex=True)
ax[0].plot(sites["stallLeft"], lw=0.8, label="stallLeft")
ax[0].plot(-sites["stallRight"], lw=0.8, label="-stallRight")
ax[0].set_ylabel("stall prob")
ax[0].legend(loc="upper right", fontsize=8)

ax[1].plot(1.0 / sites["stallDeath"], lw=0.8, label="lifetime while stalled")
ax[1].plot(1.0 / sites["death"], lw=0.8, ls="--", label="lifetime while moving")
ax[1].set_xlabel("monomer")
ax[1].set_ylabel("LEF lifetime")
ax[1].legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()

## Reference: every parameter

`SimParams` is a dataclass, so this lists every field with its current value. The
inline comments in `sim3D.py` explain each one.

In [ ]:
from dataclasses import fields

for f in fields(SimParams):
    value = getattr(params, f.name)
    if isinstance(value, (list, np.ndarray)) and len(value) > 6:
        value = "{0} of length {1}".format(type(value).__name__, len(value))
    print("{0:22s} {1}".format(f.name, value))